# MP09 — MSE and PSNR from Scratch

Expanded executable lesson. Prerequisites: earlier image-processing and array lessons. Learning goals: derive the calculation, predict edge cases, implement it, and interpret controlled real-image evidence. Learner mastery is not asserted.

## Setup and conventions
From repository root run `python -m pip install -e ".[dev]"`. Run `python scripts/metric_experiments.py` after downloading the pinned training scene with the portable quickstart. The notebook reads the committed numerical record; it does not silently download imagery. Arrays are HWC float64 for metrics.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp09_tools import mse_psnr,ssim_map


## Mathematics and worked example
Let prediction p and reference r have H rows, W columns and C bands, all co-registered in identical units. N=HWC samples. MSE=sum((p-r)^2)/N has squared intensity units; RMSE=sqrt(MSE) has intensity units. PSNR=10 log10(L²/MSE) is in decibels, where L is the declared dynamic range, not the observed maximum of each image. Our normalized numerical samples use L=1. Identical images have MSE=0 and positive-infinite PSNR; this is a special case, not a division error.

For r=[0,0,0,0] and p=[0,0,0,2], squared errors sum to 4, so MSE=1. With L=2, PSNR=10 log10(4)=6.0206 dB. Cast unsigned integers before subtraction to prevent wraparound. Scaling both images and L by the same factor leaves PSNR unchanged. Changing only L from 1 to 255 artificially adds 48.1308 dB. Doubling Gaussian noise standard deviation approximately quadruples MSE and lowers PSNR by 6.0206 dB; finite samples fluctuate.

In [ ]:
r=np.zeros((2,2,1),np.uint8); p=np.array([0,0,0,2],np.uint8).reshape(2,2,1)
mse,psnr=mse_psnr(p,r,2)
assert mse==1 and np.isclose(psnr,6.020599913)
assert np.isinf(mse_psnr(r,r)[1])
assert np.isclose(mse_psnr(p/2,r/2,1)[1],psnr)
print(mse,psnr)

## Recorded real-data experiment
The source checksum, acquisition date, license, normalization, seed and settings are in the record below. The input is a 64×64 training crop of real four-band NAIP aerial imagery; these are synthetic distortions, not satellite sensor measurements. Reproduce with `python scripts/metric_experiments.py`.

In [ ]:
record=json.loads((root/'reports/mp09-mp11-experiments.json').read_text())
print(record['source']['id'],record['source']['sha256'])
print(record['protocol'])

## Practice
Predict before running: (1) MSE for errors [1,-1]? (2) PSNR change when only L doubles? (3) Can high PSNR prove that a generated road is real?

## Worked solutions
(1) Both squared errors are one, so MSE=1. (2) +6.0206 dB, a protocol change rather than model improvement. (3) No; an average pixel metric cannot certify individual structures or compensate for invalid reference alignment.

In [ ]:
assert mse_psnr(np.array([1.,-1.]).reshape(1,2,1),np.zeros((1,2,1)))[0]==1
print('Range-table measurements:',record['range_sensitivity'])

## Figure and interpretation
See [dashboard](../../reports/mp09-mp11-dashboard.png). Blur and matched-MSE noise have equal PSNR but different SSIM. All error maps share a scale. Contrast and blur affect this low-texture crop less than the .01 noise case; this observation is not a universal ranking. Reflect: explain a failure mode that both metrics could miss.

## References and limitations
[Official scikit-image metric documentation](https://scikit-image.org/docs/stable/api/skimage.metrics.html), checked 2026-10-19, documents explicit ranges and SSIM settings. Our NumPy implementation uses declared uniform population moments and valid windows; it does not claim default-package equivalence. No alignment correction, learned perceptual metric or independent-scene claim is made.